# LMIP Secrets Registration

**Purpose**: Register and validate Databricks Secrets for LMIP platform

**Secret Scopes**:
* `lmip-api` - API keys and credentials for external data sources
* `lmip-notifications` - Email/Slack/webhook credentials for alerts
* `lmip-superset` - Superset database connection strings

---

## Setup Instructions

### 1. Create Secret Scopes (via CLI or UI)

```bash
# Using Databricks CLI
databricks secrets create-scope --scope lmip-api
databricks secrets create-scope --scope lmip-notifications
databricks secrets create-scope --scope lmip-superset
```

### 2. Add Secrets

```bash
# API credentials (if needed for future sources)
databricks secrets put --scope lmip-api --key remotive-api-key
databricks secrets put --scope lmip-api --key arbeitnow-api-key

# Notification credentials
databricks secrets put --scope lmip-notifications --key email-smtp-server
databricks secrets put --scope lmip-notifications --key email-smtp-port
databricks secrets put --scope lmip-notifications --key email-username
databricks secrets put --scope lmip-notifications --key email-password
databricks secrets put --scope lmip-notifications --key slack-webhook-url

# Superset credentials
databricks secrets put --scope lmip-superset --key superset-db-connection-string
databricks secrets put --scope lmip-superset --key superset-api-token
```

### 3. Run This Notebook

```python
result = dbutils.notebook.run("/LMIP/notebooks/init/init_register_secrets", timeout_seconds=300)
if result != "SUCCESS":
    print("Some secrets are missing - see validation report")
```

---

**Note**: This notebook validates secret references but cannot read secret values for security

In [0]:
# Databricks notebook source
from datetime import datetime, timezone

# Define required secret scopes and keys
REQUIRED_SECRETS = {
    "lmip-api": [
        # Currently public APIs, but reserved for future authenticated sources
        # "remotive-api-key",
        # "arbeitnow-api-key"
    ],
    "lmip-notifications": [
        "email-smtp-server",
        "email-smtp-port",
        "email-username",
        "email-password",
        "slack-webhook-url"
    ],
    "lmip-superset": [
        "superset-db-connection-string",
        "superset-api-token"
    ]
}

# Validation results
validation_results = []

def log_secret_check(scope, key, status, message):
    """Log secret validation result"""
    validation_results.append({
        "scope": scope,
        "key": key,
        "status": status,
        "message": message,
        "timestamp": datetime.now(timezone.utc)
    })
    symbol = "✓" if status == "PRESENT" else ("⚠" if status == "OPTIONAL" else "✗")
    print(f"{symbol} {scope}/{key}: {message}")

print("="*60)
print("LMIP SECRETS VALIDATION")
print("="*60)
print(f"Started: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}")
print()

In [0]:
# Databricks notebook source
print("\n[1] Available Secret Scopes")
print("-" * 40)

try:
    available_scopes = dbutils.secrets.listScopes()
    scope_names = [scope.name for scope in available_scopes]
    
    print(f"Found {len(scope_names)} secret scope(s):")
    for scope in scope_names:
        print(f"  • {scope}")
    
    # Check for required scopes
    for required_scope in REQUIRED_SECRETS.keys():
        if required_scope in scope_names:
            log_secret_check(required_scope, "<scope>", "PRESENT", "Scope exists")
        else:
            log_secret_check(required_scope, "<scope>", "MISSING", "Scope does not exist")
            
except Exception as e:
    print(f"✗ Error listing secret scopes: {str(e)}")

In [0]:
# Databricks notebook source
print("\n[2] Secret Key Validation")
print("-" * 40)

for scope, keys in REQUIRED_SECRETS.items():
    if not keys:  # Skip empty scope definitions
        print(f"\n{scope}: No required keys (optional scope)")
        continue
        
    print(f"\n{scope}:")
    
    try:
        # Check if scope exists
        available_scopes = [s.name for s in dbutils.secrets.listScopes()]
        
        if scope not in available_scopes:
            for key in keys:
                log_secret_check(scope, key, "MISSING", "Scope does not exist")
            continue
        
        # List keys in scope
        try:
            secret_list = dbutils.secrets.list(scope)
            available_keys = [secret.key for secret in secret_list]
            
            # Check each required key
            for key in keys:
                if key in available_keys:
                    log_secret_check(scope, key, "PRESENT", "Secret exists")
                else:
                    log_secret_check(scope, key, "MISSING", "Secret not found")
                    
        except Exception as e:
            for key in keys:
                log_secret_check(scope, key, "ERROR", f"Cannot list secrets: {str(e)[:50]}")
            
    except Exception as e:
        print(f"  ✗ Error validating scope {scope}: {str(e)}")

In [0]:
# Databricks notebook source
print("\n[3] Test Secret References")
print("-" * 40)

# Test that we can reference secrets (without exposing values)
test_secrets = [
    ("lmip-notifications", "email-smtp-server"),
    ("lmip-superset", "superset-db-connection-string")
]

for scope, key in test_secrets:
    try:
        # Try to get secret - this will work if secret exists
        # We don't print the value for security
        secret_value = dbutils.secrets.get(scope=scope, key=key)
        
        # Check if we got a value (not empty)
        if secret_value and len(secret_value) > 0:
            print(f"✓ Can retrieve {scope}/{key} (value not displayed)")
        else:
            print(f"⚠ {scope}/{key} exists but is empty")
            
    except Exception as e:
        error_msg = str(e)
        if "does not exist" in error_msg.lower():
            print(f"✗ {scope}/{key} does not exist")
        else:
            print(f"✗ {scope}/{key} error: {error_msg[:50]}")

In [0]:
# Databricks notebook source
print("\n[4] Missing Secrets Setup Script")
print("-" * 40)

# Find missing secrets
missing_secrets = [r for r in validation_results if r["status"] == "MISSING"]

if missing_secrets:
    print("\nTo create missing secrets, run these commands:\n")
    
    # Group by scope
    missing_by_scope = {}
    for result in missing_secrets:
        scope = result["scope"]
        key = result["key"]
        if key != "<scope>":  # Skip scope-level errors
            if scope not in missing_by_scope:
                missing_by_scope[scope] = []
            missing_by_scope[scope].append(key)
    
    # Generate commands
    for scope, keys in missing_by_scope.items():
        print(f"# Create scope {scope}")
        print(f"databricks secrets create-scope --scope {scope}\n")
        
        for key in keys:
            print(f"# Set {key}")
            print(f"databricks secrets put --scope {scope} --key {key}")
        print()
else:
    print("\n✓ All required secrets are present")

In [0]:
# Databricks notebook source
print("\n" + "="*60)
print("SECRETS VALIDATION SUMMARY")
print("="*60)

# Count results
present = sum(1 for r in validation_results if r["status"] == "PRESENT")
missing = sum(1 for r in validation_results if r["status"] == "MISSING")
optional = sum(1 for r in validation_results if r["status"] == "OPTIONAL")
errors = sum(1 for r in validation_results if r["status"] == "ERROR")

print(f"\n✓ Present: {present}")
print(f"✗ Missing: {missing}")
if optional > 0:
    print(f"⚠ Optional: {optional}")
if errors > 0:
    print(f"✗ Errors: {errors}")

# Determine overall status
if missing > 0 or errors > 0:
    overall_status = "INCOMPLETE"
    print("\n✗ SECRETS SETUP INCOMPLETE")
    print("\nMissing secrets:")
    for result in validation_results:
        if result["status"] in ["MISSING", "ERROR"]:
            print(f"  - {result['scope']}/{result['key']}: {result['message']}")
    print("\nRefer to the setup script above to create missing secrets")
else:
    overall_status = "SUCCESS"
    print("\n✓ ALL REQUIRED SECRETS REGISTERED")

print(f"\nCompleted: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}")
print("="*60)

# Return status for orchestration
dbutils.notebook.exit(overall_status)